# Vision-language reasoning: comparing observation modes

This notebook compares three ways of grounding the `LLMPlannerAgent`:

- **DOM-only** — the agent sees the serialized DOM, no image.
- **Screenshot-only** — the agent sees a rendered screenshot, no DOM.
- **Combined** — the agent sees both.

To keep the notebook offline and deterministic, we use a **mock VLM client**
that succeeds only when the channel a task actually needs is present. This lets
us reason about *where each mode fails* without spending API calls or launching
a browser. The same `LLMPlannerAgent`, `Runner`, and metrics from the package
are used unchanged.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import json
import pandas as pd

from agentic_qa_lab.agents import LLMMessage, LLMPlannerAgent, ObservationMode, Runner
from agentic_qa_lab.domain import ActionResult, AgentAction, Observation, TaskSpec
from agentic_qa_lab.environments import BrowserEnvironment
from agentic_qa_lab.evaluation import compute_summary


In [ ]:
# Two tasks that stress different channels:
#   - "form"  : type into a labelled field -> needs the DOM
#   - "icon"  : click a wordless icon button -> needs the screenshot
TASKS = {
    "form": TaskSpec(task_id="form", goal="Type your name into the user field",
                     start_url="https://e.com/"),
    "icon": TaskSpec(task_id="icon", goal="Click the wordless icon button",
                     start_url="https://e.com/"),
}


class FakeEnv(BrowserEnvironment):
    """Always-available env that exposes a DOM and a (dummy) screenshot path."""

    def __init__(self) -> None:
        self._step = 0

    def _obs(self) -> Observation:
        return Observation(step=self._step, url="https://e.com/", title="Demo",
                           dom_snapshot="<input id=\'user\'><button id=\'go\'></button>",
                           screenshot_path="dummy.png", timestamp=1.0 + self._step)

    def open(self, url: str) -> Observation:
        return self._obs()

    def observe(self) -> Observation:
        self._step += 1
        return self._obs()

    def execute(self, action: AgentAction) -> ActionResult:
        return ActionResult.ok()

    def close(self) -> None:
        pass


class MockVLM:
    """Succeeds only when the channel the task needs is in the prompt.

    The planner encodes the mode into each prompt: the DOM block appears only in
    DOM/combined modes, and the screenshot is attached only in
    screenshot/combined modes. The mock reads those exact signals.
    """

    def complete(self, messages: list[LLMMessage]) -> str:
        user = messages[-1]
        has_dom = "DOM (" in user.content
        has_image = bool(user.images)
        goal = user.content.split("GOAL:", 1)[1].splitlines()[0].lower()
        needs_vision = "icon" in goal
        ok = has_image if needs_vision else has_dom
        if ok:
            return json.dumps({"type": "finish", "reason": "channel available"})
        missing = "screenshot" if needs_vision else "DOM"
        return json.dumps({"type": "fail", "reason": f"cannot act without the {missing}"})


In [ ]:
def run_mode(mode: ObservationMode) -> dict:
    results = []
    for task in TASKS.values():
        agent = LLMPlannerAgent(MockVLM(), observation_mode=mode)
        results.append(Runner().run(task, agent, FakeEnv()))
    summary = compute_summary(results)
    return {
        "mode": mode.value,
        "success_rate": summary.success_rate,
        "form": next(r.status.value for r in results if r.task_id == "form"),
        "icon": next(r.status.value for r in results if r.task_id == "icon"),
    }


rows = [run_mode(mode) for mode in ObservationMode]
comparison = pd.DataFrame(rows).set_index("mode")
comparison


## Failure-case analysis

The per-task columns show *where* each mode breaks, not just the headline rate:

- **DOM-only** completes `form` (the labelled input is in the DOM) but **fails
  `icon`** — a wordless button has no text to match, so a DOM-only agent has
  nothing to ground on.
- **Screenshot-only** is the mirror image: it completes `icon` from the rendered
  layout but **fails `form`**, because reading exact field values/attributes
  from pixels is unreliable and our mock refuses to guess.
- **Combined** completes both: the DOM gives stable selectors while the
  screenshot disambiguates visual-only targets.

In a real run, the interesting failures cluster differently: screenshot-only
tends to misfire on dynamic text and offscreen elements, while DOM-only misses
canvas/SVG widgets and anything rendered without semantic markup. Combined costs
more tokens per step (the image is attached every turn) — the trade-off is
accuracy versus cost.


## Conclusion

Observation mode is a real lever on agent reliability. DOM grounding is cheap
and precise where semantic markup exists; visual grounding covers what the DOM
cannot express; combining them dominated on this toy benchmark and is the safe
default when budget allows. The package supports all three through
`LLMPlannerAgent(observation_mode=...)`, so swapping modes in a benchmark is a
one-line change — making this comparison reproducible on real tasks.
